# 03 — Calibration

Loads `outputs/predictions/val_predictions.parquet` and, for each model, fits Platt scaling and isotonic regression **on validation predictions only** (alongside the uncalibrated baseline). Computes the full calibration-metric suite and generates reliability diagrams with confidence histograms.

**Caveat (by design):** calibrators are fit and evaluated on the same validation set here — this is optimistic in-sample calibration quality. A held-out check happens in `04_final_evaluation.ipynb` on the test set, touched there for the first and only time.

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

import joblib
import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.calibration import PlattScaler, IsotonicCalibrator, compute_calibration_metrics
from src.plots import plot_reliability_diagram_with_histogram

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("03_calibration started (seed=%d)", seed)

MODELS_DIR = Path(config["training"]["models_output_dir"])
TABLES_DIR = Path(config["training"]["tables_output_dir"])
FIGURES_DIR = Path(config["calibration"]["figures_output_dir"])
METRICS_TABLE_PATH = Path(config["calibration"]["metrics_table_path"])
ECE_BINS = tuple(config["calibration"]["ece_bins"])  # (10, 15)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
val_predictions_path = Path(config["training"]["val_predictions_path"])
if not val_predictions_path.exists():
    raise FileNotFoundError(
        f"'{val_predictions_path}' not found. Run notebooks/02_train_models.ipynb first."
    )

val_predictions_df = pd.read_parquet(val_predictions_path)
model_names = sorted(val_predictions_df["model_name"].unique())
print(f"Loaded {len(val_predictions_df)} rows for models: {model_names}")

## Fit calibrators + compute metrics for every (model, method) pair

In [ ]:
metric_rows = []
calibration_objects = {}
predictions_by_method = {}  # model_name -> {method: y_prob}

for model_name in model_names:
    model_df = val_predictions_df[val_predictions_df["model_name"] == model_name]
    y_true = model_df["true_label"].to_numpy()
    y_prob_uncal = model_df["predicted_prob"].to_numpy()

    platt = PlattScaler().fit(y_prob_uncal, y_true)
    iso = IsotonicCalibrator().fit(y_prob_uncal, y_true)

    calibration_objects[f"{model_name}_platt"] = platt
    calibration_objects[f"{model_name}_isotonic"] = iso

    predictions_by_method[model_name] = {
        "uncalibrated": y_prob_uncal,
        "platt": platt.transform(y_prob_uncal),
        "isotonic": iso.transform(y_prob_uncal),
    }

    for method, y_prob in predictions_by_method[model_name].items():
        metrics = compute_calibration_metrics(y_true, y_prob, ece_bins=ECE_BINS)
        metric_rows.append({"model": model_name, "calibration_method": method, **metrics})
        logger.info(
            "[%s/%s] brier=%.4f ece10=%.4f ece15=%.4f mce=%.4f nll=%.4f intercept=%.3f slope=%.3f",
            model_name, method, metrics["brier_score"], metrics["ece_10bins"], metrics["ece_15bins"],
            metrics["mce"], metrics["nll"], metrics["calibration_intercept"], metrics["calibration_slope"],
        )

calibration_metrics_df = pd.DataFrame(metric_rows)
calibration_metrics_df

## Save calibration objects

In [ ]:
for name, obj in calibration_objects.items():
    out_path = MODELS_DIR / f"calibration_{name}.joblib"
    joblib.dump(obj, out_path)
    logger.info("Calibration object '%s' saved to %s", name, out_path)

print(f"Saved {len(calibration_objects)} calibration objects to {MODELS_DIR}")

## Table 3: calibration metrics

In [ ]:
METRICS_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
calibration_metrics_df.to_csv(METRICS_TABLE_PATH, index=False)
logger.info("Calibration metrics table saved to %s", METRICS_TABLE_PATH)
print(f"Saved {METRICS_TABLE_PATH}")

## Reliability diagrams with confidence histograms (PNG + PDF)

In [ ]:
import matplotlib.pyplot as plt

for model_name, methods in predictions_by_method.items():
    y_true = val_predictions_df.loc[val_predictions_df["model_name"] == model_name, "true_label"].to_numpy()
    for method_name, y_prob in methods.items():
        fig = plot_reliability_diagram_with_histogram(
            y_true, y_prob, n_bins=10, title=f"{model_name} \u2014 {method_name} (validation set)",
        )
        base_path = FIGURES_DIR / f"reliability_{model_name}_{method_name}"
        fig.savefig(base_path.with_suffix(".png"), dpi=300, bbox_inches="tight")
        fig.savefig(base_path.with_suffix(".pdf"), bbox_inches="tight")
        plt.close(fig)
        logger.info("Reliability diagram saved: %s.png / .pdf", base_path)

print(f"Saved {sum(len(m) for m in predictions_by_method.values())} reliability diagrams (PNG+PDF) to {FIGURES_DIR}")

## Run metadata

In [ ]:
log_run_metadata(
    seed=seed,
    extra={
        "notebook": "03_calibration",
        "n_val": int(val_predictions_df["model_name"].value_counts().iloc[0]),
        "models_calibrated": model_names,
        "methods": ["uncalibrated", "platt", "isotonic"],
        "test_labels_used": False,
    },
)
logger.info("03_calibration finished")